# v4 Alpha + 3D Object Extraction — 2×T4

**共 15 个 Cell，建议 Cell 01 → Cell 15 顺序运行。**

## v4 修复重点

v2 的 Alpha Benchmark 可用，因此 **BEN2 / ISNet / BiRefNet 路线保持不动**。

v2 的 3D Object Extraction 使用：

```text
Transformers Sam2Processor + Sam2Model
```

该路径在多 bbox / 后处理组合下出现 `/objects/extract HTTP 500`。

v3 改为官方 Grounded-SAM-2 / Meta SAM2 的标准调用：

```text
Image
  ↓
Florence-2 Large <OD> / <REGION_PROPOSAL>      cuda:0
  ↓
N × XYXY bbox
  ↓
Meta 官方 SAM2ImagePredictor                   cuda:1
  ├─ set_image(image)          ← 每张图只编码一次
  └─ predict(box=N×4)          ← 所有 bbox 一次批量分割
  ↓
N × instance mask
  ↓
3D Object Filter
  ├─ 最小面积
  ├─ IoU 去重
  └─ 高包含子物体过滤
  ↓
1024×1024 透明 PNG
```

### 额外修复

- SAM2 改用官方 `sam2` Python 包和官方 `.pt` checkpoint。
- Kaggle 使用 `SAM2_BUILD_CUDA=0`，避免不必要的 CUDA 扩展编译。
- Florence-2 强制 `attn_implementation="eager"`，规避 `_supports_sdpa` 兼容问题。
- Worker 启动时会 **真实 Warmup Florence + SAM2**，不是只加载模型。
- Florence bbox 会先做 finite / clip / 非零面积检查。
- `/objects/extract` 出错时返回具体异常信息，Gradio 不再只显示 `HTTP 500`。


### v4 关键修复

Florence-2 的官方 remote code 与新版 Transformers generation cache API 不兼容；v4 固定：

```text
transformers==4.49.0
```

并继续使用：

```text
Florence-2 Large → Meta 官方 SAM2ImagePredictor → 3D Object Filter
```

Cell 11 会通过新的 Worker 版本号自动停止旧 v3 Worker 并重启。

## Cell 清单

| Cell | 作用 |
|---|---|
| 01 | 本说明 |
| 02 | 安装依赖 + 官方 SAM2 |
| 03 | 检查 2×T4 / CUDA / SAM2 |
| 04 | `%%writefile download_models.py` |
| 05 | 下载 / 复用模型 |
| 06 | `%%writefile model_runtime.py`：原 Alpha 路线 |
| 07 | `%%writefile object_runtime.py`：Florence-2 + 官方 SAM2 |
| 08 | `%%writefile model_server.py` |
| 09 | `%%writefile app.py` |
| 10 | Worker 说明 |
| 11 | 启动 v3 Worker |
| 12 | Gradio 说明 |
| 13 | 启动 Gradio |
| 14 | Tunnel 说明 |
| 15 | `gradio-tun 7860` |


In [1]:
# Cell 02 — 安装依赖
!python -m pip install -q -U uv
!pip install gradio-tunneling
!uv pip uninstall --system onnxruntime onnxruntime-gpu || true
!uv pip install --system "pillow==11.3.0" "onnxruntime-gpu==1.21.0" "rembg[gpu]" "gradio>=5" "transformers==4.49.0" "accelerate>=1.2" "fastapi>=0.115" "uvicorn>=0.30" "python-multipart>=0.0.9" "requests>=2.32" "hydra-core>=1.3.2" "iopath>=0.1.10" timm einops kornia safetensors "git+https://github.com/PramaLLC/BEN2.git"

# 官方 Meta SAM2；Kaggle 不编译可选 CUDA 后处理扩展，避免 NVCC / ABI 问题。
!uv pip uninstall --system SAM-2 || true
!SAM2_BUILD_CUDA=0 uv pip install --system --no-deps "git+https://github.com/facebookresearch/sam2.git"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.1/20.1 MB 58.0 MB/s eta 0:00:00:00:0100:01
Using Python 3.12.13 environment at: /usr
Resolved 6 packages in 390ms                                         
Prepared 1 package in 534ms                                              
Installed 1 package in 3ms.9.0                              
 + gradio-tunneling==0.9.0
Using Python 3.12.13 environment at: /usr
Using Python 3.12.13 environment at: /usr
Resolved 118 packages in 4.16s                                       
Prepared 12 packages in 5.85s                                            
Uninstalled 3 packages in 612ms
Installed 12 packages in 51ms                               
 + ben2==0.0.1 (from git+https://github.com/PramaLLC/BEN2.git@2c99a5da477b5523585bfa5c893888a6e818a8f6)
 + coloredlogs==15.0.1
 - huggingface-hub==1.11.0
 + huggingface-hub==0.36.2
 + humanfriendly==10.0
 + hydra-core==1.3.5
 + iopath==0.1.10
 + onnxruntime-gpu==1.21.0
 + portalocker==4.3.0
 + pymatting==1.1.15
 +

In [2]:
# Cell 03 — 检查 Kaggle 2×T4 / CUDA / SAM2
import PIL, torch, onnxruntime as ort, transformers
import sam2

try: ort.preload_dlls()
except Exception as e: print("ort.preload_dlls:", e)

print("Pillow:", PIL.__version__)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()): print(f"cuda:{i}:", torch.cuda.get_device_name(i))
print("Transformers:", transformers.__version__)
print("ONNX Runtime:", ort.__version__)
print("ORT providers:", ort.get_available_providers())
print("SAM2 package:", sam2.__file__)

assert PIL.__version__ == "11.3.0"
assert transformers.__version__ == "4.49.0", f"Florence-2 需要 transformers 4.49.0，当前是 {transformers.__version__}"
assert torch.cuda.is_available(), "没有检测到 GPU"
assert torch.cuda.device_count() >= 2, f"需要 Kaggle 2×T4，目前只有 {torch.cuda.device_count()} 张 GPU"
assert "CUDAExecutionProvider" in ort.get_available_providers(), "rembg 没有 CUDAExecutionProvider"
print("✅ 2×T4 / CUDA / ONNX Runtime / SAM2 正常")


Pillow: 11.3.0
PyTorch: 2.10.0+cu128
CUDA: 12.8
GPU count: 2
cuda:0: Tesla T4
cuda:1: Tesla T4
Transformers: 4.49.0
ONNX Runtime: 1.21.0
ORT providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']
SAM2 package: /usr/local/lib/python3.12/dist-packages/sam2/__init__.py
✅ 2×T4 / CUDA / ONNX Runtime / SAM2 正常


In [3]:
%%writefile download_models.py
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from huggingface_hub import snapshot_download, hf_hub_download
from huggingface_hub.utils import disable_progress_bars

disable_progress_bars()

ROOT = Path("/kaggle/working/models")
BEN2_DIR = ROOT / "ben2"
BIREF_DIR = ROOT / "birefnet"
U2NET_HOME = ROOT / "u2net"
FLORENCE_DIR = ROOT / "florence2-large"
SAM2_DIR = ROOT / "sam2.1-hiera-large"

for p in (BEN2_DIR, BIREF_DIR, U2NET_HOME, FLORENCE_DIR, SAM2_DIR): p.mkdir(parents=True, exist_ok=True)

def dl_ben2():
    required = BEN2_DIR / "model.safetensors"
    if not required.exists(): snapshot_download("PramaLLC/BEN2", local_dir=BEN2_DIR, allow_patterns=["config.json", "model.safetensors"])
    return f"BEN2 OK -> {required}"

def dl_biref():
    required = BIREF_DIR / "model.safetensors"
    if not required.exists(): snapshot_download("ZhengPeng7/BiRefNet", local_dir=BIREF_DIR, allow_patterns=["config.json", "model.safetensors", "*.py"])
    return f"BiRefNet OK -> {required}"

def dl_isnet():
    required = U2NET_HOME / "isnet-general-use.onnx"
    if not required.exists(): hf_hub_download("jellybox/isnet-general-use", "isnet-general-use.onnx", revision="407fc6fbe11da9209fa37b128c0fbbb03f29ad54", local_dir=U2NET_HOME)
    return f"ISNet OK -> {required}"

def dl_florence():
    required = FLORENCE_DIR / "model.safetensors"
    if not required.exists(): snapshot_download("microsoft/Florence-2-large", local_dir=FLORENCE_DIR, allow_patterns=["*.json", "*.py", "*.safetensors", "*.txt", "*.model", "tokenizer*", "vocab*", "merges*"])
    return f"Florence-2 Large OK -> {required}"

def dl_sam2():
    required = SAM2_DIR / "sam2.1_hiera_large.pt"
    if not required.exists(): hf_hub_download("facebook/sam2.1-hiera-large", "sam2.1_hiera_large.pt", local_dir=SAM2_DIR)
    return f"SAM2.1 Hiera Large OK -> {required}"

with ThreadPoolExecutor(max_workers=5) as ex:
    for msg in ex.map(lambda fn: fn(), [dl_ben2, dl_biref, dl_isnet, dl_florence, dl_sam2]): print(msg)


Writing download_models.py


In [4]:
# Cell 05 — 下载 / 复用本地模型
!python download_models.py


Traceback (most recent call last):
  File "/kaggle/working/download_models.py", line 43, in <module>
    for msg in ex.map(lambda fn: fn(), [dl_ben2, dl_biref, dl_isnet, dl_florence, dl_sam2]): print(msg)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/concurrent/futures/_base.py", line 619, in result_iterator
    yield _result_or_cancel(fs.pop())
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/concurrent/futures/_base.py", line 317, in _result_or_cancel
    return fut.result(timeout)
           ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/concurrent/futures/_base.py", line 456, in result
    return self.__get_result()
           ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/concurrent/futures/_base.py", line 401, in __get_result
    raise self._exception
  File "/usr/lib/python3.12/concurrent/futures/thread.py", line 59, in run
    result = self.fn(*self.args, **self.kwargs)
             ^^^

In [5]:
%%writefile model_runtime.py
# Cell 06 — BEN2 / ISNet / BiRefNet 常驻运行时

import os
import time
import threading
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

ROOT = Path("/kaggle/working/models")
HF_HOME = ROOT / "hf"
BEN2_DIR = ROOT / "ben2"
BIREF_DIR = ROOT / "birefnet"
U2NET_HOME = ROOT / "u2net"

HF_HOME.mkdir(parents=True, exist_ok=True)
U2NET_HOME.mkdir(parents=True, exist_ok=True)

# 必须在 transformers/rembg import 之前设置。
os.environ["HF_HOME"] = str(HF_HOME)
os.environ["U2NET_HOME"] = str(U2NET_HOME)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import numpy as np
import torch
from PIL import Image
from torchvision import transforms
from transformers import AutoModelForImageSegmentation
from safetensors.torch import load_file
from ben2.modeling_ben2 import BEN_Base
from rembg import remove, new_session

if torch.cuda.device_count() < 2:
    raise RuntimeError(
        f"需要 Kaggle 2×T4，目前只检测到 {torch.cuda.device_count()} 张 GPU"
    )

BEN2_DEVICE = torch.device("cuda:0")
BIREF_DEVICE = torch.device("cuda:1")
REMBG_GPU = 0
REMBG_MODEL_NAME = "isnet-general-use"

GPU0_LOCK = threading.Lock()
GPU1_LOCK = threading.Lock()

BIREF_TRANSFORM = transforms.Compose([
    transforms.Resize((1024, 1024)),
    transforms.ToTensor(),
    transforms.Normalize([.485, .456, .406], [.229, .224, .225]),
])

def sync(device):
    torch.cuda.synchronize(device)

def prepare_image(image):
    if isinstance(image, np.ndarray):
        image = Image.fromarray(image)
    if not isinstance(image, Image.Image):
        raise TypeError(f"不支持的图片类型: {type(image)!r}")
    return image.convert("RGB")

def biref_tensor(image):
    return (
        BIREF_TRANSFORM(image.convert("RGB"))
        .unsqueeze(0)
        .to(BIREF_DEVICE, non_blocking=True)
    )

def gpu_status():
    lines = []
    for i in range(2):
        free, total = torch.cuda.mem_get_info(i)
        lines.append(
            f"cuda:{i} | {torch.cuda.get_device_name(i)} | "
            f"{(total-free)/2**30:.2f}/{total/2**30:.2f} GB"
        )
    return "\n".join(lines)

def _require(path):
    if not Path(path).exists():
        raise FileNotFoundError(
            f"缺少模型文件: {path}\n请先运行 download_models.py"
        )

_require(BEN2_DIR / "model.safetensors")
_require(BIREF_DIR / "model.safetensors")
_require(U2NET_HOME / "isnet-general-use.onnx")

# ============================================================
# Load once
# ============================================================

print("\n" + "=" * 70)
print("Loading BEN2 Base (LOCAL) -> cuda:0")
print("=" * 70)
t = time.perf_counter()

# 绕过 BEN2 AutoModel 的 model_info(repo_id) 网络查询，直接本地载入 safetensors。
BEN2_MODEL = BEN_Base()
BEN2_STATE = load_file(str(BEN2_DIR / "model.safetensors"), device="cpu")
BEN2_MODEL.load_state_dict(BEN2_STATE, strict=True)
del BEN2_STATE
BEN2_MODEL = BEN2_MODEL.to(BEN2_DEVICE).eval()
sync(BEN2_DEVICE)
print(f"BEN2 loaded: {time.perf_counter() - t:.2f}s")

print("\n" + "=" * 70)
print("Loading BiRefNet (LOCAL) -> cuda:1")
print("=" * 70)
t = time.perf_counter()
BIREF_MODEL = AutoModelForImageSegmentation.from_pretrained(
    str(BIREF_DIR),
    trust_remote_code=True,
    local_files_only=True,
).to(BIREF_DEVICE).eval()
sync(BIREF_DEVICE)
print(f"BiRefNet loaded: {time.perf_counter() - t:.2f}s")

print("\n" + "=" * 70)
print("Loading rembg/ISNet (LOCAL) -> cuda:0")
print("=" * 70)
t = time.perf_counter()
REMBG_SESSION = new_session(
    REMBG_MODEL_NAME,
    providers=[
        ("CUDAExecutionProvider", {"device_id": REMBG_GPU}),
        "CPUExecutionProvider",
    ],
)
print(f"rembg loaded: {time.perf_counter() - t:.2f}s")
print("providers:", REMBG_SESSION.inner_session.get_providers())

# BEN2 import 会主动改 cudnn 配置；模型构造完成后再恢复 benchmark。
torch.backends.cudnn.benchmark = True

# ============================================================
# Warmup once
# ============================================================

print("\n" + "=" * 70)
print("Warming up models...")
print("=" * 70)

warm = Image.new("RGB", (512, 512), (128, 128, 128))

print("Warmup BEN2...")
t = time.perf_counter()
with torch.inference_mode():
    _ = BEN2_MODEL.inference(warm)
sync(BEN2_DEVICE)
print(f"BEN2 warmup: {time.perf_counter() - t:.2f}s")

print("Warmup BiRefNet...")
x = biref_tensor(warm)
t = time.perf_counter()
with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):
    _ = BIREF_MODEL(x)[-1]
sync(BIREF_DEVICE)
del x
print(f"BiRefNet warmup: {time.perf_counter() - t:.2f}s")

print("Warmup rembg...")
t = time.perf_counter()
_ = remove(warm, session=REMBG_SESSION)
print(f"rembg warmup: {time.perf_counter() - t:.2f}s")

del warm

print("\n" + "=" * 70)
print("ALL MODELS READY — model_runtime stays resident")
print("=" * 70)
print(gpu_status())
print("=" * 70)

# ============================================================
# Inference
# ============================================================

def _run_ben2(image):
    image = prepare_image(image)
    sync(BEN2_DEVICE)
    t = time.perf_counter()
    with torch.inference_mode():
        result = BEN2_MODEL.inference(image)
    sync(BEN2_DEVICE)
    dt = time.perf_counter() - t
    result = result.convert("RGBA")
    return (
        result,
        result.getchannel("A"),
        f"BEN2 Base\nGPU: cuda:0\n推理: {dt:.3f}s\n"
        f"输入: {image.width}×{image.height}",
    )

def run_ben2(image):
    with GPU0_LOCK:
        return _run_ben2(image)

def _run_rembg(image):
    image = prepare_image(image)
    t = time.perf_counter()
    result = remove(image, session=REMBG_SESSION).convert("RGBA")
    dt = time.perf_counter() - t
    return (
        result,
        result.getchannel("A"),
        f"ISNet / rembg\nGPU: cuda:0\n推理: {dt:.3f}s\n"
        f"输入: {image.width}×{image.height}",
    )

def run_rembg(image):
    with GPU0_LOCK:
        return _run_rembg(image)

def _run_birefnet(image):
    image = prepare_image(image)
    x = biref_tensor(image)

    sync(BIREF_DEVICE)
    t = time.perf_counter()

    with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):
        pred = BIREF_MODEL(x)[-1].sigmoid()

    sync(BIREF_DEVICE)
    dt = time.perf_counter() - t

    alpha = transforms.ToPILImage()(
        pred[0].squeeze().float().cpu()
    ).resize(image.size, Image.Resampling.LANCZOS)

    del x, pred

    result = image.convert("RGBA")
    result.putalpha(alpha)

    return (
        result,
        alpha,
        f"BiRefNet\nGPU: cuda:1\n推理: {dt:.3f}s\n"
        f"模型输入: 1024×1024\n原图: {image.width}×{image.height}",
    )

def run_birefnet(image):
    with GPU1_LOCK:
        return _run_birefnet(image)

def _gpu0(image):
    with GPU0_LOCK:
        return _run_ben2(image), _run_rembg(image)

def _gpu1(image):
    with GPU1_LOCK:
        return _run_birefnet(image)

def compare_all(image):
    image = prepare_image(image)
    t = time.perf_counter()

    with ThreadPoolExecutor(max_workers=2) as ex:
        f0 = ex.submit(_gpu0, image)
        f1 = ex.submit(_gpu1, image)
        ben2, rembg = f0.result()
        biref = f1.result()

    return (
        ben2,
        rembg,
        biref,
        f"总耗时: {time.perf_counter() - t:.3f}s\n\n{gpu_status()}",
    )


Writing model_runtime.py


In [6]:
%%writefile object_runtime.py
import time, threading
from pathlib import Path

import numpy as np
import torch
from PIL import Image, ImageDraw
from transformers import AutoProcessor, AutoModelForCausalLM
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

ROOT = Path("/kaggle/working/models")
FLORENCE_DIR = ROOT / "florence2-large"
SAM2_DIR = ROOT / "sam2.1-hiera-large"
SAM2_CHECKPOINT = SAM2_DIR / "sam2.1_hiera_large.pt"
SAM2_CONFIG = "configs/sam2.1/sam2.1_hiera_l.yaml"

FLORENCE_DEVICE = torch.device("cuda:0")
SAM_DEVICE = torch.device("cuda:1")
OBJECT_LOCK = threading.Lock()

if torch.cuda.device_count() < 2: raise RuntimeError(f"需要 Kaggle 2×T4，目前只有 {torch.cuda.device_count()} 张 GPU")
if not (FLORENCE_DIR / "model.safetensors").exists(): raise FileNotFoundError("Florence-2 权重不存在，请先运行 Cell 05")
if not SAM2_CHECKPOINT.exists(): raise FileNotFoundError("SAM2.1 checkpoint 不存在，请先运行 Cell 05")

def sync(device): torch.cuda.synchronize(device)

print("\n" + "=" * 70)
print("Loading Florence-2 Large (LOCAL) -> cuda:0")
t = time.perf_counter()
FLORENCE_PROCESSOR = AutoProcessor.from_pretrained(str(FLORENCE_DIR), trust_remote_code=True, local_files_only=True)
FLORENCE_MODEL = AutoModelForCausalLM.from_pretrained(str(FLORENCE_DIR), trust_remote_code=True, torch_dtype=torch.float16, attn_implementation="eager", local_files_only=True).to(FLORENCE_DEVICE).eval()
sync(FLORENCE_DEVICE)
print(f"Florence-2 loaded: {time.perf_counter() - t:.2f}s")

print("\n" + "=" * 70)
print("Loading Meta SAM 2.1 Hiera Large (OFFICIAL) -> cuda:1")
t = time.perf_counter()
SAM2_MODEL = build_sam2(SAM2_CONFIG, str(SAM2_CHECKPOINT), device=str(SAM_DEVICE), mode="eval", apply_postprocessing=False)
SAM_PREDICTOR = SAM2ImagePredictor(SAM2_MODEL)
sync(SAM_DEVICE)
print(f"SAM2.1 loaded: {time.perf_counter() - t:.2f}s")

def gpu_status():
    lines = []
    for i in range(2):
        free, total = torch.cuda.mem_get_info(i)
        lines.append(f"cuda:{i} | {torch.cuda.get_device_name(i)} | {(total-free)/2**30:.2f}/{total/2**30:.2f} GB")
    return "\n".join(lines)

def run_florence(image, mode="od"):
    task = "<OD>" if mode == "od" else "<REGION_PROPOSAL>"
    inputs = FLORENCE_PROCESSOR(text=task, images=image, return_tensors="pt").to(FLORENCE_DEVICE, torch.float16)
    with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):
        generated = FLORENCE_MODEL.generate(input_ids=inputs["input_ids"], pixel_values=inputs["pixel_values"], max_new_tokens=1024, early_stopping=False, do_sample=False, num_beams=3)
    generated_text = FLORENCE_PROCESSOR.batch_decode(generated, skip_special_tokens=False)[0]
    parsed = FLORENCE_PROCESSOR.post_process_generation(generated_text, task=task, image_size=(image.width, image.height)).get(task, {})
    boxes, labels = parsed.get("bboxes", []), parsed.get("labels", [])
    if len(labels) != len(boxes): labels = [""] * len(boxes)
    labels = [str(label).strip() or f"region_{i+1:03d}" for i, label in enumerate(labels)]
    return boxes, labels

def sanitize_proposals(image, boxes, labels):
    clean_boxes, clean_labels = [], []
    for box, label in zip(boxes, labels):
        try:
            b = np.asarray(box, dtype=np.float32).reshape(4)
        except Exception:
            continue
        if not np.isfinite(b).all(): continue
        x1, y1, x2, y2 = b.tolist()
        x1, x2 = sorted((max(0.0, min(float(image.width - 1), x1)), max(0.0, min(float(image.width), x2))))
        y1, y2 = sorted((max(0.0, min(float(image.height - 1), y1)), max(0.0, min(float(image.height), y2))))
        if x2 - x1 < 2 or y2 - y1 < 2: continue
        clean_boxes.append([x1, y1, x2, y2])
        clean_labels.append(label)
    return clean_boxes, clean_labels

def run_sam(image, boxes):
    if not boxes: return np.empty((0, image.height, image.width), dtype=bool), np.empty((0,), dtype=np.float32)
    input_boxes = np.asarray(boxes, dtype=np.float32).reshape(-1, 4)
    image_np = np.asarray(image.convert("RGB"), dtype=np.uint8)
    with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):
        SAM_PREDICTOR.set_image(image_np)
        masks, scores, _ = SAM_PREDICTOR.predict(point_coords=None, point_labels=None, box=input_boxes, multimask_output=False)
    SAM_PREDICTOR.reset_predictor()
    masks = np.asarray(masks)
    scores = np.asarray(scores)
    if masks.ndim == 4 and masks.shape[1] == 1: masks = masks[:, 0]
    elif masks.ndim == 2: masks = masks[None, ...]
    if masks.ndim != 3: raise RuntimeError(f"SAM2 返回异常 mask shape: {masks.shape}")
    if masks.shape[0] != len(input_boxes): raise RuntimeError(f"SAM2 mask 数量 {masks.shape[0]} != bbox 数量 {len(input_boxes)}")
    return masks.astype(bool), scores.reshape(-1)

def mask_iou(a, b):
    union = np.logical_or(a, b).sum()
    return 0.0 if union == 0 else float(np.logical_and(a, b).sum() / union)

def filter_objects(boxes, labels, masks, scores=None, min_area_ratio=0.005, containment=0.92, duplicate_iou=0.88, max_objects=24):
    if len(masks) == 0: return []
    image_area = masks.shape[-2] * masks.shape[-1]
    candidates = []
    for i, (box, label, mask) in enumerate(zip(boxes, labels, masks)):
        area = int(mask.sum())
        if area / image_area < min_area_ratio: continue
        score = float(scores[i]) if scores is not None and i < len(scores) else None
        candidates.append({"source_index": i, "label": label, "bbox": [float(v) for v in box], "mask": mask, "area": area, "area_ratio": area / image_area, "sam_score": score})
    candidates.sort(key=lambda x: x["area"], reverse=True)
    deduped = []
    for obj in candidates:
        if any(mask_iou(obj["mask"], kept["mask"]) >= duplicate_iou for kept in deduped): continue
        deduped.append(obj)
    kept = []
    for i, obj in enumerate(deduped):
        contained = False
        for larger in deduped[:i]:
            inside = np.logical_and(obj["mask"], larger["mask"]).sum() / max(obj["area"], 1)
            if inside >= containment and larger["area"] >= obj["area"] * 1.35:
                contained = True
                break
        if not contained: kept.append(obj)
        if len(kept) >= max_objects: break
    return kept

def square_crop(image, mask, padding=0.12, size=1024):
    ys, xs = np.where(mask)
    if len(xs) == 0: return Image.new("RGBA", (size, size), (0, 0, 0, 0))
    x1, x2, y1, y2 = xs.min(), xs.max() + 1, ys.min(), ys.max() + 1
    w, h = x2 - x1, y2 - y1
    side = max(8, int(np.ceil(max(w, h) * (1 + 2 * padding))))
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    left, top = int(np.floor(cx - side / 2)), int(np.floor(cy - side / 2))
    right, bottom = left + side, top + side
    sx1, sy1, sx2, sy2 = max(0, left), max(0, top), min(image.width, right), min(image.height, bottom)
    rgba = image.convert("RGBA")
    rgba.putalpha(Image.fromarray(mask.astype(np.uint8) * 255, mode="L"))
    canvas = Image.new("RGBA", (side, side), (0, 0, 0, 0))
    canvas.paste(rgba.crop((sx1, sy1, sx2, sy2)), (sx1 - left, sy1 - top))
    return canvas.resize((size, size), Image.Resampling.LANCZOS)

def make_preview(image, objects):
    arr = np.asarray(image.convert("RGB"), dtype=np.float32).copy()
    colors = [(255,99,71),(64,158,255),(65,200,120),(255,190,60),(180,100,255),(255,90,180),(60,210,210),(220,220,70)]
    for i, obj in enumerate(objects):
        mask, color = obj["mask"], np.asarray(colors[i % len(colors)], dtype=np.float32)
        arr[mask] = arr[mask] * 0.68 + color * 0.32
    preview = Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))
    draw = ImageDraw.Draw(preview)
    for i, obj in enumerate(objects):
        x1, y1, x2, y2 = obj["bbox"]
        color = colors[i % len(colors)]
        label = f"{i+1:02d} {obj['label']}"
        draw.rectangle((x1, y1, x2, y2), outline=color, width=max(2, int(min(image.size) / 350)))
        tb = draw.textbbox((0, 0), label)
        tw, th = tb[2] - tb[0], tb[3] - tb[1]
        top = max(0, y1 - th - 8)
        draw.rectangle((x1, top, x1 + tw + 8, y1), fill=color)
        draw.text((x1 + 4, top + 2), label, fill=(0, 0, 0))
    return preview

def extract_objects(image, mode="od", min_area_ratio=0.005, containment=0.92, padding=0.12, max_objects=24):
    image = image.convert("RGB")
    with OBJECT_LOCK:
        t0 = time.perf_counter()
        raw_boxes, raw_labels = run_florence(image, mode)
        boxes, labels = sanitize_proposals(image, raw_boxes, raw_labels)
        t1 = time.perf_counter()
        masks, scores = run_sam(image, boxes)
        t2 = time.perf_counter()
    objects = filter_objects(boxes, labels, masks, scores=scores, min_area_ratio=min_area_ratio, containment=containment, max_objects=max_objects)
    preview, packed = make_preview(image, objects), []
    for i, obj in enumerate(objects, start=1):
        mask_img = Image.fromarray(obj["mask"].astype(np.uint8) * 255, mode="L")
        crop = square_crop(image, obj["mask"], padding=padding, size=1024)
        packed.append({"id": i, "label": obj["label"], "bbox": obj["bbox"], "mask_area": obj["area"], "area_ratio": obj["area_ratio"], "sam_score": obj["sam_score"], "mask": mask_img, "crop": crop})
    info = {"mode": mode, "raw_proposals": len(raw_boxes), "proposals": len(boxes), "kept": len(packed), "florence_s": round(t1 - t0, 3), "sam_s": round(t2 - t1, 3), "total_s": round(time.perf_counter() - t0, 3)}
    return preview, packed, info

# 真正执行两套模型的启动自检，避免等到 Gradio 请求时才发现 HTTP 500。
print("\n" + "=" * 70)
print("Warming up Florence-2 + official SAM2...")
warm_image = Image.new("RGB", (256, 256), (128, 128, 128))
t = time.perf_counter()
_ = run_florence(warm_image, "od")
print(f"Florence warmup OK: {time.perf_counter() - t:.2f}s")

warm_box = [[32.0, 32.0, 224.0, 224.0]]
t = time.perf_counter()
warm_masks, warm_scores = run_sam(warm_image, warm_box)
assert warm_masks.shape == (1, 256, 256), f"SAM2 warmup shape 异常: {warm_masks.shape}"
print(f"SAM2 warmup OK: {time.perf_counter() - t:.2f}s | score={float(warm_scores[0]):.4f}")
del warm_image, warm_masks, warm_scores
print("✅ 3D Object runtime READY")
print(gpu_status())
print("=" * 70)


Writing object_runtime.py


In [7]:
%%writefile model_server.py
import base64, traceback
from io import BytesIO

from fastapi import FastAPI, File, Form, HTTPException, UploadFile
from PIL import Image

from model_runtime import compare_all, run_ben2, run_birefnet, run_rembg
from object_runtime import extract_objects, gpu_status

PIPELINE_VERSION = "v4-alpha+florence2-4.49+sam2.1-official"
app = FastAPI(title="Alpha + 3D Object Worker")

def read_image(file):
    try: return Image.open(file.file).convert("RGB")
    except Exception as e: raise HTTPException(status_code=400, detail=f"图片读取失败: {type(e).__name__}: {e}")

def encode_png(image):
    buf = BytesIO(); image.save(buf, format="PNG"); return base64.b64encode(buf.getvalue()).decode("ascii")

def pack_alpha(result):
    rgba, alpha, info = result
    return {"rgba": encode_png(rgba), "alpha": encode_png(alpha), "info": info}

@app.get("/health")
def health(): return {"ready": True, "version": PIPELINE_VERSION, "gpu_status": gpu_status()}

@app.post("/infer/ben2")
def infer_ben2(file: UploadFile = File(...)): return pack_alpha(run_ben2(read_image(file)))

@app.post("/infer/rembg")
def infer_rembg(file: UploadFile = File(...)): return pack_alpha(run_rembg(read_image(file)))

@app.post("/infer/birefnet")
def infer_birefnet(file: UploadFile = File(...)): return pack_alpha(run_birefnet(read_image(file)))

@app.post("/compare")
def compare(file: UploadFile = File(...)):
    ben2, rembg, biref, status = compare_all(read_image(file))
    return {"ben2": pack_alpha(ben2), "rembg": pack_alpha(rembg), "birefnet": pack_alpha(biref), "status": status}

@app.post("/objects/extract")
def objects_extract(file: UploadFile = File(...), mode: str = Form("od"), min_area_ratio: float = Form(0.005), containment: float = Form(0.92), padding: float = Form(0.12), max_objects: int = Form(24)):
    try:
        preview, objects, info = extract_objects(read_image(file), mode=mode, min_area_ratio=min_area_ratio, containment=containment, padding=padding, max_objects=max_objects)
        return {"preview": encode_png(preview), "objects": [{"id": o["id"], "label": o["label"], "bbox": o["bbox"], "mask_area": o["mask_area"], "area_ratio": o["area_ratio"], "sam_score": o["sam_score"], "mask": encode_png(o["mask"]), "crop": encode_png(o["crop"])} for o in objects], "info": info}
    except HTTPException:
        raise
    except Exception as e:
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=f"{type(e).__name__}: {e}") from e

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="127.0.0.1", port=7861, workers=1, log_level="info")


Writing model_server.py


In [20]:
%%writefile app.py
import base64, html, json, math, time, uuid
from io import BytesIO
from pathlib import Path
import gradio as gr
import requests
from PIL import Image

WORKER = "http://127.0.0.1:7861"
TIMEOUT = 300
ALPHA_ROOT = Path("/kaggle/working/alpha_outputs")
OBJECT_ROOT = Path("/kaggle/working/object_outputs")
ALPHA_ROOT.mkdir(parents=True, exist_ok=True)
OBJECT_ROOT.mkdir(parents=True, exist_ok=True)

def worker_status():
    try:
        r = requests.get(f"{WORKER}/health", timeout=3); r.raise_for_status(); d = r.json()
        return f"✅ {d.get('version', 'worker')} READY\n{d.get('gpu_status', '')}"
    except Exception as e: return f"❌ Worker NOT READY\n{e}"

def normalize_path(file):
    if isinstance(file, (str, Path)): return Path(file)
    if hasattr(file, "name"): return Path(file.name)
    if hasattr(file, "path"): return Path(file.path)
    raise TypeError(f"未知文件类型: {type(file)}")

def image_bytes(image):
    buf = BytesIO(); image.convert("RGB").save(buf, format="PNG"); return buf.getvalue()

def decode_png(data): return Image.open(BytesIO(base64.b64decode(data))).copy()

def call_compare(image):
    r = requests.post(f"{WORKER}/compare", files={"file": ("input.png", image_bytes(image), "image/png")}, timeout=TIMEOUT); r.raise_for_status(); return r.json()

def call_objects(image, mode, min_area_pct, containment, padding_pct, max_objects):
    data = {"mode": "od" if mode.startswith("主物体") else "region", "min_area_ratio": float(min_area_pct) / 100, "containment": float(containment), "padding": float(padding_pct) / 100, "max_objects": int(max_objects)}
    r = requests.post(f"{WORKER}/objects/extract", data=data, files={"file": ("input.png", image_bytes(image), "image/png")}, timeout=TIMEOUT)
    if not r.ok:
        try: detail = r.json().get("detail", r.text)
        except Exception: detail = r.text
        raise RuntimeError(f"Object Worker HTTP {r.status_code}: {detail}")
    return r.json()

def slug(text):
    s = "".join(c if c.isalnum() or c in "-_" else "_" for c in str(text)).strip("_")
    return s[:48] or "object"

def save_json_atomic(path, data):
    tmp = path.with_suffix(path.suffix + ".tmp"); tmp.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8"); tmp.replace(path)

def alpha_save_payload(payload, output_dir, name):
    rgba_path, alpha_path = output_dir / f"{name}.png", output_dir / f"{name}_alpha.png"
    decode_png(payload["rgba"]).save(rgba_path); decode_png(payload["alpha"]).save(alpha_path)
    return {"image": str(rgba_path), "alpha": str(alpha_path), "info": payload.get("info", "")}

def alpha_process(files, page_size, progress=gr.Progress()):
    if not files: raise gr.Error("请先上传至少一张图片")
    run_dir = ALPHA_ROOT / (time.strftime("run_%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:6]); run_dir.mkdir(parents=True, exist_ok=True)
    records, total = [], len(files)
    for i, f in enumerate(files, 1):
        progress((i - 1, total), desc=f"Alpha {i}/{total}"); path = normalize_path(f); item_dir = run_dir / f"{i:04d}_{slug(path.stem)}"; item_dir.mkdir(parents=True, exist_ok=True); record = {"index": i, "name": path.name}
        try:
            with Image.open(path) as im: image = im.convert("RGB")
            input_path = item_dir / "input.png"; image.save(input_path); result = call_compare(image)
            record.update({"input": str(input_path), "ben2": alpha_save_payload(result["ben2"], item_dir, "ben2"), "isnet": alpha_save_payload(result["rembg"], item_dir, "isnet"), "birefnet": alpha_save_payload(result["birefnet"], item_dir, "birefnet")})
        except Exception as e: record["error"] = f"{type(e).__name__}: {e}"
        records.append(record); save_json_atomic(run_dir / "manifest.json", {"run_dir": str(run_dir), "records": records})
    progress((total, total), desc="完成")
    return records, 1, alpha_render(records, 1, page_size), alpha_page_label(records, 1, page_size), f"{worker_status()}\n\nAlpha 图片：{len(records)}\n保存：{run_dir}", str(run_dir)

def image_data_uri(path, max_side=640):
    path = Path(path)
    if not path.exists(): return ""
    with Image.open(path) as img:
        img = img.convert("RGBA"); img.thumbnail((max_side, max_side), Image.Resampling.LANCZOS); buf = BytesIO(); img.save(buf, format="PNG", optimize=True)
    return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode("ascii")

def alpha_figure(title, path, info=""):
    if not path or not Path(path).exists(): return f'<div class="model-result"><div class="model-title">{html.escape(title)}</div><div class="missing">无输出</div></div>'
    info_html = html.escape(info).replace("\n", "<br>")
    return f'<div class="model-result"><div class="model-title">{html.escape(title)}</div><div class="image-stage"><img src="{image_data_uri(path)}" loading="lazy"></div>{f"""<div class="model-info">{info_html}</div>""" if info else ""}</div>'

def alpha_render(records, page, page_size):
    if not records: return '<div class="empty">暂无 Alpha 结果</div>'
    page_size = int(page_size); total_pages = max(1, math.ceil(len(records) / page_size)); page = max(1, min(int(page), total_pages)); start = (page - 1) * page_size; cards = []
    for r in records[start:start + page_size]:
        if "error" in r: cards.append(f'<article class="sample-card"><div class="sample-header">#{r["index"]:03d} {html.escape(r["name"])}</div><div class="error">{html.escape(r["error"])}</div></article>'); continue
        figures = alpha_figure("Original", r["input"]) + alpha_figure("BEN2", r["ben2"]["image"], r["ben2"]["info"]) + alpha_figure("ISNet", r["isnet"]["image"], r["isnet"]["info"]) + alpha_figure("BiRefNet", r["birefnet"]["image"], r["birefnet"]["info"])
        cards.append(f'<article class="sample-card"><div class="sample-header">#{r["index"]:03d}　{html.escape(r["name"])}</div><div class="compare-grid">{figures}</div></article>')
    return '<div class="results-container">' + "".join(cards) + "</div>"

def alpha_page_label(records, page, page_size):
    if not records: return "0 / 0"
    page_size = int(page_size); pages = max(1, math.ceil(len(records) / page_size)); page = max(1, min(int(page), pages)); start = (page - 1) * page_size + 1; end = min(page * page_size, len(records))
    return f"第 {page} / {pages} 页 · {start}–{end} / {len(records)}"

def alpha_prev(records, page, page_size):
    page = max(1, int(page) - 1); return page, alpha_render(records, page, page_size), alpha_page_label(records, page, page_size)

def alpha_next(records, page, page_size):
    if not records: return 1, alpha_render([], 1, page_size), "0 / 0"
    pages = max(1, math.ceil(len(records) / int(page_size))); page = min(pages, int(page) + 1); return page, alpha_render(records, page, page_size), alpha_page_label(records, page, page_size)

def alpha_resize(records, page_size): return 1, alpha_render(records, 1, page_size), alpha_page_label(records, 1, page_size)

def object_save_scene(run_dir, index, source_path, result):
    scene_dir = run_dir / f"{index:04d}_{slug(source_path.stem)}"; scene_dir.mkdir(parents=True, exist_ok=True)
    with Image.open(source_path) as im: source = im.convert("RGB")
    source_file, preview_file = scene_dir / "source.png", scene_dir / "preview.png"; source.save(source_file); decode_png(result["preview"]).save(preview_file)
    objects = []
    for obj in result["objects"]:
        obj_dir = scene_dir / f"object_{obj['id']:03d}_{slug(obj['label'])}"; obj_dir.mkdir(parents=True, exist_ok=True)
        crop_file, mask_file = obj_dir / "crop.png", obj_dir / "mask.png"; decode_png(obj["crop"]).save(crop_file); decode_png(obj["mask"]).save(mask_file)
        meta = {"id": obj["id"], "label": obj["label"], "bbox": obj["bbox"], "mask_area": obj["mask_area"], "area_ratio": obj["area_ratio"], "sam_score": obj.get("sam_score"), "crop": str(crop_file), "mask": str(mask_file)}
        save_json_atomic(obj_dir / "metadata.json", meta); objects.append(meta)
    record = {"index": index, "name": source_path.name, "source": str(source_file), "preview": str(preview_file), "objects": objects, "selected_ids": [o["id"] for o in objects], "info": result["info"], "scene_dir": str(scene_dir)}
    save_json_atomic(scene_dir / "objects.json", {"info": result["info"], "objects": objects, "selected_ids": record["selected_ids"]})
    return record

def object_process(files, mode, min_area_pct, containment, padding_pct, max_objects, progress=gr.Progress()):
    if not files: raise gr.Error("请先上传至少一张图片")
    run_dir = OBJECT_ROOT / (time.strftime("run_%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:6]); run_dir.mkdir(parents=True, exist_ok=True)
    records, total = [], len(files)
    for i, f in enumerate(files, 1):
        progress((i - 1, total), desc=f"3D Object {i}/{total}"); path = normalize_path(f)
        try:
            with Image.open(path) as im: image = im.convert("RGB")
            records.append(object_save_scene(run_dir, i, path, call_objects(image, mode, min_area_pct, containment, padding_pct, max_objects)))
        except Exception as e: records.append({"index": i, "name": path.name, "error": f"{type(e).__name__}: {e}"})
        save_json_atomic(run_dir / "manifest.json", {"run_dir": str(run_dir), "records": records})
    progress((total, total), desc="完成")
    preview, gallery, table, choices, page = object_render(records, 0)
    total_objects = sum(len(r.get("objects", [])) for r in records)
    return records, 0, preview, gallery, table, choices, page, f"{worker_status()}\n\n图片：{len(records)} | 物体：{total_objects}\n保存：{run_dir}", str(run_dir)

def object_render(records, idx):
    if not records: return None, [], [], gr.update(choices=[], value=[]), "0 / 0"
    idx = max(0, min(int(idx), len(records) - 1)); r = records[idx]
    if "error" in r: return None, [], [["ERROR", r["error"], "", ""]], gr.update(choices=[], value=[]), f"{idx+1} / {len(records)} · {r['name']}"
    preview = Image.open(r["preview"]).convert("RGB").copy()
    gallery = [(Image.open(o["crop"]).convert("RGBA").copy(), f"#{o['id']:02d}  {o['label']}") for o in r["objects"]]
    table = [[o["id"], o["label"], f"{o['area_ratio']*100:.2f}%", str([round(x, 1) for x in o["bbox"]])] for o in r["objects"]]
    labels = [f"{o['id']:02d} | {o['label']}" for o in r["objects"]]
    selected = {int(x) for x in r.get("selected_ids", [])}
    selected_labels = [label for label, obj in zip(labels, r["objects"]) if obj["id"] in selected]
    info = r.get("info", {}); page = f"{idx+1} / {len(records)} · {r['name']} · proposals {info.get('proposals', 0)} → kept {info.get('kept', 0)} · {info.get('total_s', 0)}s"
    return preview, gallery, table, gr.update(choices=labels, value=selected_labels), page
    
def object_prev(records, idx):
    idx = max(0, int(idx) - 1); return idx, *object_render(records, idx)

def object_next(records, idx):
    idx = min(max(0, len(records) - 1), int(idx) + 1) if records else 0; return idx, *object_render(records, idx)

def object_save_selection(records, idx, selected_labels):
    if not records: return records, "没有结果"
    idx = max(0, min(int(idx), len(records) - 1)); r = records[idx]
    if "error" in r: return records, "当前图片处理失败"
    selected_ids = []
    for label in selected_labels or []:
        try: selected_ids.append(int(str(label).split("|", 1)[0].strip()))
        except Exception: pass
    r["selected_ids"] = selected_ids
    save_json_atomic(Path(r["scene_dir"]) / "objects.json", {"info": r["info"], "objects": r["objects"], "selected_ids": selected_ids})
    return records, f"✅ 已保存选择：{len(selected_ids)} / {len(r['objects'])} 个物体"

CSS = """
.gradio-container{max-width:1600px!important;margin:0 auto!important}
.results-container{display:flex;flex-direction:column;gap:20px}.sample-card{border:1px solid rgba(128,128,128,.22);border-radius:16px;overflow:hidden}.sample-header{padding:12px 15px;font-weight:650;border-bottom:1px solid rgba(128,128,128,.18)}
.compare-grid{display:grid;grid-template-columns:repeat(4,minmax(0,1fr))}.model-result{min-width:0;border-right:1px solid rgba(128,128,128,.14)}.model-result:last-child{border-right:none}.model-title{padding:9px 11px;font-weight:650;border-bottom:1px solid rgba(128,128,128,.14)}
.image-stage{height:320px;display:flex;align-items:center;justify-content:center;padding:8px;background-color:#f8f8f8;background-image:linear-gradient(45deg,#e8e8e8 25%,transparent 25%),linear-gradient(-45deg,#e8e8e8 25%,transparent 25%),linear-gradient(45deg,transparent 75%,#e8e8e8 75%),linear-gradient(-45deg,transparent 75%,#e8e8e8 75%);background-size:20px 20px;background-position:0 0,0 10px,10px -10px,-10px 0}.image-stage img{width:100%;height:100%;object-fit:contain}.model-info{padding:8px 11px;font-size:11px;opacity:.65;font-family:ui-monospace,monospace}.empty{padding:120px;text-align:center;opacity:.55}
@media(max-width:1100px){.compare-grid{grid-template-columns:repeat(2,minmax(0,1fr))}}@media(max-width:700px){.compare-grid{grid-template-columns:1fr}}
"""

with gr.Blocks(title="v2 Alpha + 3D Object Extraction", css=CSS) as demo:
    gr.Markdown("# v2 Alpha + 3D Object Extraction\n同一个常驻 Worker，两条独立路线：**Alpha Benchmark** 与 **Florence-2 → SAM2.1 → 3D Object Crop**。")
    with gr.Tabs():
        with gr.Tab("Alpha Benchmark"):
            alpha_records, alpha_page = gr.State([]), gr.State(1)
            with gr.Row():
                with gr.Column(scale=3):
                    alpha_files = gr.File(label="批量上传图片", file_count="multiple", file_types=["image"], type="filepath")
                    alpha_run = gr.Button("运行 BEN2 / ISNet / BiRefNet", variant="primary")
                with gr.Column(scale=2):
                    alpha_status = gr.Textbox(label="Worker / 状态", value=worker_status(), lines=6, interactive=False)
                    alpha_dir = gr.Textbox(label="Kaggle 保存目录", interactive=False)
            with gr.Row():
                alpha_prev_btn = gr.Button("← 上一页"); alpha_page_text = gr.Textbox(value="0 / 0", show_label=False, interactive=False, scale=3); alpha_page_size = gr.Dropdown([4, 8, 12], value=4, label="每页"); alpha_next_btn = gr.Button("下一页 →")
            alpha_html = gr.HTML('<div class="empty">暂无 Alpha 结果</div>')
            alpha_run.click(alpha_process, [alpha_files, alpha_page_size], [alpha_records, alpha_page, alpha_html, alpha_page_text, alpha_status, alpha_dir])
            alpha_prev_btn.click(alpha_prev, [alpha_records, alpha_page, alpha_page_size], [alpha_page, alpha_html, alpha_page_text])
            alpha_next_btn.click(alpha_next, [alpha_records, alpha_page, alpha_page_size], [alpha_page, alpha_html, alpha_page_text])
            alpha_page_size.change(alpha_resize, [alpha_records, alpha_page_size], [alpha_page, alpha_html, alpha_page_text])

        with gr.Tab("3D Object Extraction"):
            object_records, object_idx = gr.State([]), gr.State(0)
            with gr.Row():
                with gr.Column(scale=3):
                    object_files = gr.File(label="批量上传图片", file_count="multiple", file_types=["image"], type="filepath")
                    object_run = gr.Button("识别并拆分独立物体", variant="primary")
                with gr.Column(scale=2):
                    object_status = gr.Textbox(label="Worker / 状态", value=worker_status(), lines=6, interactive=False)
                    object_dir = gr.Textbox(label="Kaggle 保存目录", interactive=False)
            with gr.Accordion("3D Object Filter", open=False):
                with gr.Row():
                    object_mode = gr.Radio(["主物体（推荐）", "尽量找全"], value="主物体（推荐）", label="Florence 模式")
                    min_area = gr.Slider(0.1, 5.0, value=0.5, step=0.1, label="最小 Mask 面积 %")
                    containment = gr.Slider(0.70, 0.99, value=0.92, step=0.01, label="子物体包含阈值")
                    padding = gr.Slider(0, 30, value=12, step=1, label="Crop Padding %")
                    max_objects = gr.Slider(1, 32, value=24, step=1, label="最多保留物体")
            with gr.Row():
                object_prev_btn = gr.Button("← 上一张"); object_page = gr.Textbox(value="0 / 0", show_label=False, interactive=False, scale=4); object_next_btn = gr.Button("下一张 →")
            with gr.Row():
                object_preview = gr.Image(label="Florence + SAM2.1 实例预览", type="pil", scale=5)
                object_gallery = gr.Gallery(label="独立 1024×1024 透明 PNG", columns=3, rows=2, height=520, object_fit="contain", scale=5)
            object_table = gr.Dataframe(headers=["ID", "Label", "Mask Area", "BBox"], datatype=["number", "str", "str", "str"], interactive=False, label="最终保留物体")
            object_choices = gr.CheckboxGroup(label="真正保留 / 后续送入 3D 的物体（默认全选）")
            save_selection_btn = gr.Button("保存当前选择")
            selection_status = gr.Textbox(show_label=False, interactive=False)
            object_run.click(object_process, [object_files, object_mode, min_area, containment, padding, max_objects], [object_records, object_idx, object_preview, object_gallery, object_table, object_choices, object_page, object_status, object_dir])
            object_prev_btn.click(object_prev, [object_records, object_idx], [object_idx, object_preview, object_gallery, object_table, object_choices, object_page])
            object_next_btn.click(object_next, [object_records, object_idx], [object_idx, object_preview, object_gallery, object_table, object_choices, object_page])
            save_selection_btn.click(object_save_selection, [object_records, object_idx, object_choices], [object_records, selection_status])

if __name__ == "__main__":
    demo.queue(default_concurrency_limit=1).launch(server_name="127.0.0.1", server_port=7860, share=False, show_error=True)

Overwriting app.py


## Cell 10 — 启动常驻 v4 Worker

运行下一格 **Cell 11**。

v4 Worker 在启动成功前会完成：

1. Alpha 三模型加载 / warmup。
2. Florence-2 加载。
3. 官方 SAM2.1 加载。
4. Florence-2 实际 generate warmup。
5. SAM2 `set_image + box predict` 实际 warmup。

因此如果 Cell 11 打印 `v4 Worker READY`，说明 3D Object 的核心模型路径已经真正执行过，而不是仅仅“模型加载成功”。


In [27]:
# Cell 11 — 启动 / 复用 v2 Worker
import subprocess, sys, time
from pathlib import Path
import requests

WORKER_URL = "http://127.0.0.1:7861/health"
EXPECTED_VERSION = "v4-alpha+florence2-4.49+sam2.1-official"
WORKER_LOG_PATH = Path("/kaggle/working/model_server.log")

def health():
    try:
        r = requests.get(WORKER_URL, timeout=2)
        return r.json() if r.ok else {}
    except Exception: return {}

current = health()
if current.get("ready") and current.get("version") == EXPECTED_VERSION:
    print("✅ 复用现有 v4 Worker；不会重新加载模型")
else:
    if current: print("检测到旧 Worker，切换到 v4...")
    subprocess.run(["pkill", "-f", "[m]odel_server.py"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(1)
    WORKER_LOG = open(WORKER_LOG_PATH, "w", buffering=1)
    MODEL_SERVER_PROCESS = subprocess.Popen([sys.executable, "model_server.py"], stdout=WORKER_LOG, stderr=subprocess.STDOUT, cwd="/kaggle/working")
    while True:
        if MODEL_SERVER_PROCESS.poll() is not None:
            WORKER_LOG.flush(); print(WORKER_LOG_PATH.read_text(errors="replace")[-12000:]); raise RuntimeError(f"model_server.py 启动失败 code={MODEL_SERVER_PROCESS.returncode}")
        current = health()
        if current.get("ready") and current.get("version") == EXPECTED_VERSION: break
        time.sleep(1)
    print("✅ v4 Worker READY")

print(current.get("gpu_status", ""))


✅ v4 Worker READY
cuda:0 | Tesla T4 | 3.94/14.56 GB
cuda:1 | Tesla T4 | 3.74/14.56 GB


## Cell 12 — 启动 / 重启 Gradio

运行下一格 **Cell 13**。

这里只重启 `app.py:7860`，不会停止 `model_server.py:7861`。

如果 3D Object 请求仍发生异常，v4 会直接把 Worker 的具体异常类型和信息返回到 Gradio，而不是只显示 `HTTP 500`。


In [28]:
# Cell 13 — 启动 / 重启 Gradio UI
import subprocess, sys, time
from pathlib import Path
import requests

APP_LOG_PATH = Path("/kaggle/working/app.log")
subprocess.run(["pkill", "-f", "[a]pp.py"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(0.5)

APP_LOG = open(APP_LOG_PATH, "w", buffering=1)
APP_PROCESS = subprocess.Popen([sys.executable, "app.py"], stdout=APP_LOG, stderr=subprocess.STDOUT, cwd="/kaggle/working")

while True:
    if APP_PROCESS.poll() is not None:
        APP_LOG.flush(); print(APP_LOG_PATH.read_text(errors="replace")[-12000:]); raise RuntimeError(f"app.py 启动失败 code={APP_PROCESS.returncode}")
    try:
        if requests.get("http://127.0.0.1:7860", timeout=2).ok: break
    except Exception: pass
    time.sleep(0.5)

print("✅ Gradio READY: http://127.0.0.1:7860")
print("✅ v4 Worker 仍常驻 7861")


✅ Gradio READY: http://127.0.0.1:7860
✅ v4 Worker 仍常驻 7861


## Cell 14 — 打开公网 Tunnel

最后运行 **Cell 15**。这个命令会占住当前 Cell，这是正常现象。


In [ ]:
!curl -sSf https://get.openziti.io/install.bash | sudo bash -s zrok2

In [16]:
!zrok2 enable bUrdvGnUDLQo

]10;?\[   3.544]    INFO main.(*enableCommand).run the zrok environment was successfully enabled...


In [29]:
!zrok2 share public http://127.0.0.1:7860

]10;?\]11;?\╭──────────────────────────────────────────────────╮╭──────────────────────────╮
│           gjh7bpdhgez3.shares.zrok.io            ││     [PUBLIC] [PROXY]     │
╰──────────────────────────────────────────────────╯╰──────────────────────────╯
╭──────────────────────────────────────────────────────────────────────────────╮
│                                                                              │
│                                                                              │
│                                                                              │
│                                                                              │
│                                                                              │
│                                                                              │
│                                                                              │
│                                                                              │
│           